In [1]:
import os
from joblib import cpu_count
from threadpoolctl import threadpool_limits

# Detect available logical CPU cores
TOTAL_CPUS = cpu_count(only_physical_cores=False)

# Reserve two CPUs for the operating system, if possible
N_JOBS = max(1, TOTAL_CPUS - 7)

print(f"Available CPUs: {TOTAL_CPUS}")
print(f"CPUs allocated: {N_JOBS}")

Available CPUs: 32
CPUs allocated: 25


n2p2 dataset might look like,

```python
begin                                       --- begining of one atomic configuration
lattice 5.0 0.0 0.0
lattice 0.0 5.0 0.0
lattice 0.0 0.0 5.0
atom 0.0 0.0 0.0 Si 0.0 0.0 0.0 0.0 0.0
atom 2.5 2.5 2.5 Si 0.0 0.0 0.0 0.0 0.0
energy -10.5
charge 0.0
end                                         --- end of one atomic configuration
```

Task -- getting atomic structures as ASE objects and energy values separately

In [2]:
from pathlib import Path # class 'Path' that helps locating the files
from ase import Atoms # class 'Atoms' represents atomic structures
import numpy as np
from dscribe.descriptors import ACSF # library for generating descriptors of atomic structures
from joblib import Parallel, delayed

'Atom Object' is like,

```python
atoms = Atoms(
    symbols=["Fe", "Fe"],
    positions=[[0, 0, 0], [2, 2, 2]]
)
```

They can store elements, coordinates, unit cell dimensions, periodic boundry conditions

# TASK 1 - Extract data from the data file 

Internal Helper Functions 

+ not called directly by the user
+ supports the main function

_ at the beginning indicates that this is intended to be an internal helper function

In [3]:
def _parse_n2p2_block(block): # Internal Helper function to parse a single n2p2 block and extract atomic structure and energy information from the lines in n2p2 file
    symbols = []                # block - lines belongs to one atomic configuration
    positions = []
    cell = []
    energy = None


    for line in block:
        parts = line.split() # separate the strings in a line based on whitespace

        if not parts:
            continue # if the list parts is empty, skip to next line

        if parts[0] == "lattice":
            cell.append([float(x) for x in parts[1:4]]) # Make the vector with direction values of [x,y,z] ex - from "lattice 0.0 5.0 0.0" extract [0.0, 5.0, 0.0]
            has_cell = True # lattice information for this block is found successfully

        elif parts[0] == "atom":
            positions.append([float(x) for x in parts[1:4]]) # poistion of the atom to the list positions, ex - from "atom 0.0 0.0 0.0 H" extract [0.0, 0.0, 0.0]
            symbols.append(parts[4]) # Type of atom to the list symbols, ex - from "atom 0.0 0.0 0.0 Si" extract "Si"
                                       # ignores all the other information in the line, like charge, mass, etc. 

        elif parts[0] == "energy":
            energy = float(parts[1]) # extract the reference energy value

    if energy is None:
        raise ValueError("Energy missing in n2p2 block") # raising an error if energy is not found in the block

   # if not symbols:
        #raise ValueError("No atoms found in n2p2 block") # raising an error if no atoms are found in the block

    atoms = Atoms( #Converst extracted information into ASE Atoms object
        symbols=symbols,
        positions=positions,
        cell=cell if len(cell) == 3 else None,
        pbc=(len(cell) == 3) 
    )

    return atoms, energy

A 'block' looks like this
 
```python
block = [
    "lattice 5.0 0.0 0.0",
    "lattice 0.0 5.0 0.0",
    "lattice 0.0 0.0 5.0",
    "atom 0.0 0.0 0.0 Si 0.0 0.0 0.0 0.0 0.0",
    "energy -10.5"
]
```

<img src="images/cell example.png" width="300">

In [4]:
def read_n2p2(path, n_jobs=N_JOBS): # function to read the entire dataset from n2p2 file

    current_block = []
    blocks = [] # temporarily stores the lines of one configuration
    inside = False # tracks whether Python is currently reading a configuration

    with open(path, "r") as f: # 'as file' assignss the opened file to the variable 'file' which can be used to read the file 
        for line in f:
            line = line.strip() #.strip() method removes whitespace at the beginning and end of each line, including newline characters

            if line == "begin": # when encounters 'begin', it knows a new atomic configuration is starting
                current_block = []
                inside = True # program is now reading a configuration

            elif line == "end" and inside:
                blocks.append(current_block)
                inside = False

            elif inside:
                current_block.append(line)

    results = Parallel(
        n_jobs=n_jobs,
        backend="loky"
    )(
        delayed(_parse_n2p2_block)(block)
        for block in blocks
    )

    atoms_list = [item[0] for item in results]
    energies = np.array([item[1] for item in results])

    return atoms_list, energies


atoms_list, energies = read_n2p2(
    "input.data",
    n_jobs=N_JOBS
)

print("Number of structures:", len(atoms_list))
print("Energy shape:", energies.shape)

Number of structures: 51702
Energy shape: (51702,)


# TASK 2 - Building descriptors

Task - convert those atomic structures into numerical features using Atom-Centered Symmetry Functions (ACSF)

Descriptor - numerical representation that describes the local atomic environment, including the chemical species, spatial arrangement of neighboring atoms

ACSF

- local environment surrounding an atom using mathematical functions based on distances and angles

- provides features that are designed to be invariant to translation and rotation, and to permutations of equivalent atoms

- rotating or translating a structure should not change its potential energy


### Gaussian Radial Symmetry Function

A typical expression is:

$$
G_i^2 = \sum_{j \neq i} e^{-\eta(R_{ij}-R_s)^2} f_c(R_{ij})
$$

#### Where:

| Symbol | Meaning |
|--------|---------|
| $i$ | Central atom |
| $j$ | Neighboring atom |
| $R_{ij}$ | Distance between atoms $i$ and $j$ |
| $\eta$ | Controls the width of the Gaussian function |
| $R_s$ | Shifts the center of the Gaussian |
| $f_c$ | Cutoff function |


### Gaussian Angular Symmetry Function

consider a central atom \(i\) and two neighboring atoms \(j\) and \(k\), The angular descriptor captures information about the angle θ between neighboring atoms.

<img src="images/Angular symmetry functions.png" width="300">

### Averaging atomic descriptors


Consider the following matrix:

###### Example Atomic Descriptor Matrix

| Atom | Feature 1 | Feature 2 | Feature 3 |
|------|-----------|-----------|-----------|
| Atom 1 | 2 | 4 | 6 |
| Atom 2 | 4 | 6 | 8 |
| Atom 3 | 6 | 8 | 10 |
| **Average** | **4** | **6** | **8** |

`axis=0` calculates the mean down each column, across all atoms.


```python
np.mean(atomic_descriptors, axis=0)
```

**Result:**

```python
[4.0, 6.0, 8.0]
```

This is now one feature vector representing the entire structure.


Mathematically:

$$
\overline{G}_k = \frac{1}{N}\sum_{i=1}^{N}G_{ik}
$$

Where:

- $N$ = Number of atoms in the structure.
- $G_{ik}$ = Descriptor feature $k$ for atom $i$.
- $\overline{G}_k$ = Average descriptor feature $k$.

#### Why Average?

Different atomic structures may contain different numbers of atoms.

For example, one structure might contain 10 atoms and another 20 atoms.

Averaging converts both into fixed length vectors, making them suitable for a standard neural network.

limitation: averaging loses some information about individual atomic environments. In addition, it does not explicitly account for the number of atoms, which matters when predicting total energy rather than energy per atom.

this method is convenient for a simple task, but it is not equivalent to a full atom centered MLIP.


In [5]:

import numpy as np
from dscribe.descriptors import ACSF
from joblib import Parallel, delayed

species = sorted({
    symbol
    for atoms in atoms_list
    for symbol in atoms.get_chemical_symbols()
})

acsf = ACSF(
    species=species,
    r_cut=6.0,
    g2_params=[
        [1.0, 0.0],
        [1.0, 2.0],
        [1.0, 4.0]
    ],
    g4_params=[
        [1.0, 1.0, 1.0],
        [1.0, 1.0, -1.0]
    ]
)


def calculate_atomic_descriptors(atoms):
    # One descriptor vector for EACH atom
    return np.asarray(
        acsf.create(atoms, n_jobs=1),
        dtype=np.float32
    )


def build_descriptors(atoms_list, n_jobs=N_JOBS):

    # PARALLEL: independent structures
    descriptors = Parallel(
        n_jobs=n_jobs,
        backend="loky"
    )(
        delayed(calculate_atomic_descriptors)(atoms)
        for atoms in atoms_list
    )

    return descriptors


X = build_descriptors(
    atoms_list,
    n_jobs=N_JOBS
)

# Atomic numbers identify the element of each atom
Z = [
    atoms.get_atomic_numbers().astype(np.int64)
    for atoms in atoms_list
]

energies = np.asarray(energies, dtype=np.float32)

print("Number of structures:", len(X))
print("First structure descriptor shape:", X[0].shape)
print("First structure atomic numbers:", Z[0])
print("Number of ACSF features:", acsf.get_number_of_features())


Number of structures: 51702
First structure descriptor shape: (216, 6)
First structure atomic numbers: [5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5
 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5
 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5
 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5
 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5
 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5]
Number of ACSF features: 6


# TASK 3 - Train a Neural Network

independent variables - the feature matrix containing ACSF descriptors
dependent variable - the reference energies

```text

                    Train/Test Data Split
                    (80% Train, 20% Test)
                           |
                           v
                    StandardScaler
                    (Feature Scaling)
                           |
                           v
                     MLPRegressor
                (Neural Network Training)
                           |
                           v
                  Energy Prediction
                           |
                           v
        Trained Model + Scaler + Test Predictions
```
                


X, y will look like as follows:


| Structure | Descriptor vector (X) | Energy (y) |
|-----------|------------------------|------------|
| 1 | [0.2, 0.4, 0.6] | -10.5 |
| 2 | [0.3, 0.5, 0.7] | -11.2 |
| 3 | [0.4, 0.6, 0.8] | -12.3 |
| 4 | [0.5, 0.7, 0.9] | -13.1 |

After splitting data,

| Variable | Contains |
|----------|----------|
| `X_train` | ACSF descriptors used for training |
| `X_test` | ACSF descriptors used for testing |
| `y_train` | Reference energies used for training |
| `y_test` | Reference energies used for testing |

For example, if we have 100 structures and 20 features:

```python
X_train.shape  # (80, 20) - two dimensional matrix
X_test.shape   # (20, 20)

y_train.shape  # (80,) - one dimentional array
y_test.shape   # (20,)

##### Scaling


The mathematical formula is:

$$
z = \frac{x-\mu}{\sigma}
$$

Where:

- $x$ = original feature value.
- $\mu$ = mean of that feature in the training dataset.
- $\sigma$ = standard deviation of that feature in the training dataset.
- $z$ = standardized feature value.

ex:

```python
[10, 20, 30]
```

mean is:

$$
\mu = \frac{10+20+30}{3} = 20
$$

population standard deviation is approximately:

$$
\sigma = \sqrt{\frac{(10-20)^2+(20-20)^2+(30-20)^2}{3}}
= 8.165
$$

The standardized values become approximately:

```python
[10, 20, 30] ---> [-1.225, 0.0, 1.225]
```


##### ReLu Activation Function - Rectified Linear Unit

Introduces nonlinearity into the network, enabling it to learn more complex relationships than a simple linear regression

- If use linear relationships, it allows the model to predict negative energy values as well


Its mathematical expression is:

$$
f(x) = \max(0,x)
$$

This means:

$$
f(x) =
\begin{cases}
0 & x \leq 0 \\
x & x > 0
\end{cases}
$$



##### Adam - optimization algorithm (Adaptive Moment Estimation)

- Neural network contains adjustable parameters called weights and biases.

- The objective is to find values for these parameters that minimize the prediction error.

- Adam updates these parameters iteratively using gradient information and adaptive learning rates.


In simplified form:

$$
w_{\text{new}} = w_{\text{old}} - \text{update}
$$


- Adam calculates the update using estimates of the first and second moments of the gradients.

- It is commonly used because it performs well for many neural network training problems.

In [ ]:

# ============================================================
# TASK 3: ATOM-WISE NEURAL NETWORK TRAINING
# Parallel CPU computing + Early stopping
# ============================================================

import os
import copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1. CPU PARALLELIZATION
# ============================================================

# N_JOBS is defined in the initial setup
N_THREADS = max(1, N_JOBS)

# PyTorch uses multiple CPU threads for numerical operations
torch.set_num_threads(N_THREADS)

# Inter-operation parallelism
# May already be initialized in a notebook, so handle safely
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

DEVICE = torch.device("cpu")

torch.manual_seed(42)
np.random.seed(42)

print(f"Device: {DEVICE}")
print(f"CPU threads: {torch.get_num_threads()}")


# ============================================================
# 2. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# Split by structure, not by individual atoms
indices = np.arange(len(X))

train_val_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.2,
    random_state=42
)

print(f"Training structures: {len(train_idx)}")
print(f"Validation structures: {len(val_idx)}")
print(f"Testing structures: {len(test_idx)}")


# ============================================================
# 3. DESCRIPTOR SCALING
# ============================================================

# Combine training atoms ONLY for fitting the scaler
X_train_atoms = np.concatenate(
    [X[i] for i in train_idx],
    axis=0
)

scaler = StandardScaler()
scaler.fit(X_train_atoms)

# Scale descriptors, keeping each atom separate
X_scaled = [
    scaler.transform(x).astype(np.float32)
    for x in X
]

print("Descriptor scaling completed.")


# ============================================================
# 4. ENERGY NORMALIZATION
# ============================================================

# Energy per atom is used for numerical normalization only.
# Atomic descriptors are NOT averaged.

train_energy_per_atom = np.array([
    energies[i] / len(Z[i])
    for i in train_idx
])

energy_mean = float(train_energy_per_atom.mean())
energy_std = float(train_energy_per_atom.std())

if energy_std < 1e-12:
    energy_std = 1.0

print(f"Energy mean: {energy_mean:.6f} eV/atom")
print(f"Energy std: {energy_std:.6f} eV/atom")


# ============================================================
# 5. DATASET CLASS
# ============================================================

class AtomicDataset(Dataset):

    def __init__(self, indices, X_scaled, Z, energies):
        self.indices = list(indices)
        self.X_scaled = X_scaled
        self.Z = Z
        self.energies = energies

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):

        i = self.indices[index]

        descriptors = torch.tensor(
            self.X_scaled[i],
            dtype=torch.float32
        )

        atomic_numbers = torch.tensor(
            self.Z[i],
            dtype=torch.long
        )

        energy = torch.tensor(
            self.energies[i],
            dtype=torch.float32
        )

        return descriptors, atomic_numbers, energy


# Structures may contain different numbers of atoms
def collate_structures(batch):

    descriptors, atomic_numbers, energies_batch = zip(*batch)

    return (
        list(descriptors),
        list(atomic_numbers),
        torch.stack(energies_batch)
    )


train_dataset = AtomicDataset(
    train_idx, X_scaled, Z, energies
)

val_dataset = AtomicDataset(
    val_idx, X_scaled, Z, energies
)

test_dataset = AtomicDataset(
    test_idx, X_scaled, Z, energies
)

BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_structures
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_structures
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_structures
)


# ============================================================
# 6. ATOMIC NEURAL NETWORK
# ============================================================

class AtomicNN(nn.Module):

    def __init__(self, n_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(-1)


# ============================================================
# 7. ELEMENT-SPECIFIC ATOMIC NETWORKS
# ============================================================

class AtomwisePotential(nn.Module):

    def __init__(self, n_features, elements):
        super().__init__()

        self.networks = nn.ModuleDict({
            str(int(z)): AtomicNN(n_features)
            for z in elements
        })

    def forward(self, descriptors, atomic_numbers):

        # Store one energy contribution for every atom
        atomic_energies = torch.zeros(
            len(atomic_numbers),
            dtype=descriptors.dtype,
            device=descriptors.device
        )

        for element, network in self.networks.items():

            mask = atomic_numbers == int(element)

            if mask.any():
                atomic_energies[mask] = network(
                    descriptors[mask]
                )

        # Total structure energy = sum of atomic contributions
        return atomic_energies.sum()


elements = sorted(set(
    int(z)
    for numbers in Z
    for z in numbers
))

n_features = X[0].shape[1]

model = AtomwisePotential(
    n_features=n_features,
    elements=elements
).to(DEVICE)

print("Atomic elements:", elements)
print(model)


# ============================================================
# 8. OPTIMIZER AND TRAINING SETTINGS
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.MSELoss()

MAX_EPOCHS = 500
PATIENCE = 30
MIN_DELTA = 1e-6

best_val_loss = float("inf")
best_model_state = None
best_epoch = 0
epochs_without_improvement = 0

train_history = []
val_history = []


# ============================================================
# 9. HELPER FUNCTION: ONE TRAINING / VALIDATION EPOCH
# ============================================================

def run_epoch(loader, training=True):

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_samples = 0

    for descriptors, atomic_numbers, targets in loader:

        if training:
            optimizer.zero_grad()

        predictions = []
        target_residuals = []

        with torch.set_grad_enabled(training):

            for x, z, energy in zip(
                descriptors,
                atomic_numbers,
                targets
            ):

                x = x.to(DEVICE)
                z = z.to(DEVICE)
                energy = energy.to(DEVICE)

                # Predict sum of atomic contributions
                pred = model(x, z)

                n_atoms = len(z)

                # Normalized reference energy per atom
                target_residual = (
                    energy - energy_mean * n_atoms
                ) / (energy_std * n_atoms)

                predictions.append(pred / n_atoms)
                target_residuals.append(target_residual)

            predictions = torch.stack(predictions)
            target_residuals = torch.stack(target_residuals)

            loss = criterion(
                predictions,
                target_residuals
            )

            if training:
                loss.backward()
                optimizer.step()

        batch_size = len(targets)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


# ============================================================
# 10. TRAINING LOOP WITH EARLY STOPPING
# ============================================================

print("\nStarting neural network training...\n")

for epoch in range(MAX_EPOCHS):

    # Training loss
    train_loss = run_epoch(
        train_loader,
        training=True
    )

    # Validation loss
    val_loss = run_epoch(
        val_loader,
        training=False
    )

    train_history.append(train_loss)
    val_history.append(val_loss)

    # Save the best model
    if val_loss < best_val_loss - MIN_DELTA:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

    # Print every 50 epochs
    if (epoch + 1) % 50 == 0:

        print(
            f"Epoch {epoch+1}/{MAX_EPOCHS} | "
            f"Train Loss: {train_loss:.8f} | "
            f"Val Loss: {val_loss:.8f}"
        )

    # Stop when validation loss stops improving
    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch+1}"
        )

        break


# ============================================================
# 11. RESTORE BEST MODEL
# ============================================================

model.load_state_dict(best_model_state)

print("\nTraining completed!")
print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.8f}")
print(f"Total epochs executed: {len(train_history)}")


# ============================================================
# 12. TRAINING AND VALIDATION LOSS PLOT
# ============================================================

plt.figure(figsize=(9, 5))

plt.plot(
    range(1, len(train_history) + 1),
    train_history,
    label="Training Loss"
)

plt.plot(
    range(1, len(val_history) + 1),
    val_history,
    label="Validation Loss"
)

plt.axvline(
    best_epoch,
    color="red",
    linestyle="--",
    label=f"Best Epoch ({best_epoch})"
)

plt.xlabel("Epoch")
plt.ylabel("Normalized MSE Loss")
plt.title("Atom-wise Neural Network Training")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 13. SAVE TRAINED MODEL
# ============================================================

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "elements": elements,
        "n_features": n_features,
        "energy_mean": energy_mean,
        "energy_std": energy_std,
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "train_history": train_history,
        "val_history": val_history
    },
    "atomwise_mlip_model.pth"
)

print("Model saved as atomwise_mlip_model.pth")


Device: cpu
CPU threads: 25
Training structures: 33088
Validation structures: 8273
Testing structures: 10341
Descriptor scaling completed.
Energy mean: -6.040615 eV/atom
Energy std: 0.327426 eV/atom
Atomic elements: [5]
AtomwisePotential(
  (networks): ModuleDict(
    (5): AtomicNN(
      (network): Sequential(
        (0): Linear(in_features=6, out_features=64, bias=True)
        (1): ReLU()
        (2): Linear(in_features=64, out_features=32, bias=True)
        (3): ReLU()
        (4): Linear(in_features=32, out_features=1, bias=True)
      )
    )
  )
)

Starting neural network training...



KeyboardInterrupt: 

# TASK 4

In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt


def predict_structures(model, indices, X_scaled, Z):

    model.eval()
    predictions = []

    with torch.no_grad():

        for i in indices:

            x = torch.tensor(
                X_scaled[i],
                dtype=torch.float32,
                device=DEVICE
            )

            z = torch.tensor(
                Z[i],
                dtype=torch.long,
                device=DEVICE
            )

            residual = model(x, z)

            # Convert back to total energy in eV
            total_energy = (
                residual.item() * energy_std
                + energy_mean * len(z)
            )

            predictions.append(total_energy)

    return np.array(predictions)


y_test = energies[test_idx]

y_pred = predict_structures(
    model,
    test_idx,
    X_scaled,
    Z
)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE: {mae:.6f} eV")
print(f"RMSE: {rmse:.6f} eV")

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.7)

limits = [
    min(y_test.min(), y_pred.min()),
    max(y_test.max(), y_pred.max())
]

plt.plot(limits, limits, "k--")
plt.xlabel("Actual energy (eV)")
plt.ylabel("Predicted energy (eV)")
plt.title("Atom-wise Neural Network Predictions")
plt.grid(True)
plt.tight_layout()
plt.show()


# Task 5

In [ ]:

import numpy as np
import torch
from joblib import Parallel, delayed


def prepare_new_structure(atoms):

    # Calculate ALL atomic descriptors
    atomic_features = acsf.create(
        atoms,
        n_jobs=1
    )

    atomic_features = scaler.transform(
        atomic_features
    ).astype(np.float32)

    atomic_numbers = atoms.get_atomic_numbers()

    return atomic_features, atomic_numbers


def predict_new_structures(
    new_atoms_list,
    n_jobs=N_JOBS
):

    # PARALLEL descriptor generation
    prepared = Parallel(
        n_jobs=n_jobs,
        backend="loky"
    )(
        delayed(prepare_new_structure)(atoms)
        for atoms in new_atoms_list
    )

    model.eval()
    predictions = []

    with torch.no_grad():

        for features, numbers in prepared:

            x = torch.tensor(
                features,
                dtype=torch.float32,
                device=DEVICE
            )

            z = torch.tensor(
                numbers,
                dtype=torch.long,
                device=DEVICE
            )

            residual = model(x, z)

            total_energy = (
                residual.item() * energy_std
                + energy_mean * len(z)
            )

            predictions.append(total_energy)

    return np.array(predictions)


# Example: predict the first 10 structures
new_structures = atoms_list[:10]

predicted_energies = predict_new_structures(
    new_structures,
    n_jobs=N_JOBS
)

print("Predicted total energies (eV):")
print(predicted_energies)
